# Anomaly Detection Prototype

Goal: detect unusual behavior in existing feature or risk series before building a production anomaly detector.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd

from ai_models.risk_detector import run_systemic_risk_detector

risk = run_systemic_risk_detector(
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
)
risk.tail()

,Date,RiskScore,RiskLevel,RiskFlags,Explanation
1250,2026-03-13,33.918968,Low,Correlation spike,"Primary drivers: cross-asset correlation, rate..."
1251,2026-03-16,32.106244,Low,Correlation spike,"Primary drivers: cross-asset correlation, rate..."
1252,2026-03-17,26.996667,Low,Correlation spike,"Primary drivers: cross-asset correlation, draw..."
1253,2026-03-18,36.541900,Moderate,Correlation spike,"Primary drivers: cross-asset correlation, draw..."
1254,2026-03-19,30.596162,Low,Correlation spike,"Primary drivers: cross-asset correlation, draw..."


In [3]:
risk = risk.copy()
risk["RiskZScore"] = (
    (risk["RiskScore"] - risk["RiskScore"].rolling(60, min_periods=20).mean())
    / risk["RiskScore"].rolling(60, min_periods=20).std()
)
risk["IsAnomaly"] = risk["RiskZScore"].abs() >= 2.5
risk[risk["IsAnomaly"]].tail(20)

,Date,RiskScore,RiskLevel,RiskFlags,Explanation,RiskZScore,IsAnomaly
632,2023-09-25,59.599108,Moderate,Yield inversion; Drawdown acceleration; Correl...,"Primary drivers: yield inversion, cross-asset ...",2.501153,True
633,2023-09-26,64.381901,Moderate,Yield inversion; Drawdown acceleration; Correl...,"Primary drivers: drawdown acceleration, yield ...",2.863247,True
720,2024-01-31,51.677540,Moderate,Yield inversion; Correlation spike; Rate shock,"Primary drivers: yield inversion, cross-asset ...",2.629471,True
764,2024-04-04,48.588195,Moderate,Yield inversion; Correlation spike,"Primary drivers: yield inversion, cross-asset ...",2.587267,True
771,2024-04-15,60.747965,Moderate,Yield inversion; Drawdown acceleration; Correl...,"Primary drivers: yield inversion, cross-asset ...",3.933273,True
772,2024-04-16,65.150768,Elevated,Yield inversion; Drawdown acceleration; Correl...,"Primary drivers: yield inversion, rate shock",3.943013,True
774,2024-04-18,60.345407,Moderate,Yield inversion; Drawdown acceleration; Correl...,"Primary drivers: drawdown acceleration, yield ...",2.812120,True
840,2024-07-24,56.792106,Moderate,Yield inversion; Drawdown acceleration,"Primary drivers: yield inversion, drawdown acc...",2.636786,True
847,2024-08-02,71.644225,Elevated,Yield inversion; Correlation spike; Rate shock,"Primary drivers: yield inversion, rate shock",3.642144,True
848,2024-08-05,88.708203,Elevated,Volatility spike; Yield inversion; Drawdown ac...,"Primary drivers: drawdown acceleration, yield ...",4.393911,True


In [4]:
# Optional if scikit-learn is installed.
try:
    from sklearn.ensemble import IsolationForest

    X = risk[["RiskScore"]].dropna()
    clf = IsolationForest(random_state=42, contamination=0.05)
    pred = clf.fit_predict(X)
    detected = X.copy()
    detected["AnomalyFlag"] = pred
    display(detected[detected["AnomalyFlag"] == -1].tail(20))
except ModuleNotFoundError:
    print("Install scikit-learn to test IsolationForest-based anomaly detection.")

Install scikit-learn to test IsolationForest-based anomaly detection.
